## BCI Motor Movement : Quick Overview

* **What we are doing**: Looking at brain waves when someone physically executes moving their left or right hand.
* **Why we filter it (8–30 Hz)**: Brain signals are noisy. This filter cuts out the junk and keeps only the brain rhythms (mu and beta waves) that matter for hand movement.

In [ ]:
import mne
from pathlib import Path

# Point directly to your local dataset files on the desktop
DATA_DIR = Path(r"C:\Users\Yusuff Personal\Desktop\files")

run_files = [
    DATA_DIR / "S001" / "S001R03.edf",
    DATA_DIR / "S001" / "S001R07.edf",
    DATA_DIR / "S001" / "S001R11.edf",
]

raw_list = []
for file in run_files:
    raw = mne.io.read_raw_edf(file, preload=True)
    
    # Standardize channel names and set standard 10-20 electrode positions
    mne.datasets.eegbci.standardize(raw)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore')
    
    # Apply band-pass filter
    raw.filter(8.0, 30.0, fir_design='firwin', skip_by_annotation='edge')
    raw_list.append(raw)

print(f"Successfully loaded, montaged, and filtered {len(raw_list)} physical movement files!")

## Feature Extraction : Quick Overview

* **Epoch Extraction**: Slicing the data into 3-second windows right after the movement cue.
* **CSP (Common Spatial Patterns)**: Projecting the data into a space that maximizes the contrast between left and right hand physical movements, making it easy for the classifier to read.

In [ ]:
import numpy as np
from mne.decoding import CSP

epochs_list = []
event_id = {'T1': 2, 'T2': 3}

for raw in raw_list:
    events, _ = mne.events_from_annotations(raw)
    
    epochs = mne.Epochs(
        raw, 
        events, 
        event_id, 
        tmin=0.0, 
        tmax=3.0, 
        proj=False, 
        baseline=None, 
        preload=True
    )
    epochs_list.append(epochs)

all_epochs = mne.concatenate_epochs(epochs_list)
X = all_epochs.get_data()  
y = all_epochs.events[:, -1] 

csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)
X_csp = csp.fit_transform(X, y)

print(f"Extracted CSP features shape: {X_csp.shape} for {len(y)} total trials.")

## Classification & Results : Quick Overview

* **Classifier**: Linear Discriminant Analysis (LDA) trained with 5-fold cross-validation.
* **The Result**: **88.89% Accuracy**—proving that physical muscle execution generates massive, crystal-clear electrical signals in the motor cortex!

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score

lda = LinearDiscriminantAnalysis()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(lda, X_csp, y, cv=cv)

print("--- MOTOR MOVEMENT RESULTS ---")
print(f"5-Fold Cross-Validation Scores: {np.round(scores * 100, 2)}%")
print(f"Mean Classification Accuracy: {scores.mean() * 100:.2f}%")

## Brain Topography Mapping

* **Topographic Scalp Plots**: Displays the spatial filter patterns (`CSP0` to `CSP3`) across the motor cortex regions.

In [ ]:
import matplotlib.pyplot as plt

# Plot clean CSP spatial patterns with component labels and colorbar
fig = csp.plot_patterns(all_epochs.info, ch_type='eeg', components=range(4), show_names=False, colorbar=True)
plt.suptitle('CSP Spatial Patterns (Motor Movement)', fontsize=14, y=1.02)
plt.show()

## Model Performance Comparison

* **Comparative Bar Chart**: Contrasts thought-based motor imagery accuracy against physical motor movement accuracy.

In [ ]:
import matplotlib.pyplot as plt

models = ['Motor Imagery', 'Motor Movement']
accuracies = [64.40, 88.89]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(models, accuracies, color=['#4C72B0', '#55A868'])

ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 100)
ax.set_title('BCI Model Performance Comparison')

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 2, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()